In [2]:
import pandas as pd

def tratar_vacina(caminho_arquivo, nome_vacina):
    df = pd.read_csv(caminho_arquivo, encoding="latin1", sep=";")
    df.columns = df.columns.str.strip()
    
    if "Total" in df.columns:
        df = df.drop(columns=["Total"])
    
    df[["codigo_uf", "uf"]] = df["Unidade da Federação"].str.split(n=1, expand=True)
    df = df.drop(columns=["Unidade da Federação"])
    
    anos = [col for col in df.columns if col.strip().isdigit()]
    
    df_long = df.melt(
        id_vars=["codigo_uf", "uf"],
        value_vars=anos,
        var_name="ano",
        value_name="cobertura"
    )
    
    df_long["cobertura"] = (
        df_long["cobertura"]
        .astype(str)
        .str.replace(",", ".")
        .astype(float)
    )
    
    df_long["ano"] = df_long["ano"].astype(int)
    df_long["vacina"] = nome_vacina
    
    return df_long

In [3]:
df_bcg = tratar_vacina("../data/raw/bcg.csv", "BCG")
df_poliomielite = tratar_vacina("../data/raw/poliomielite.csv", "Poliomielite")
df_triplice_viral = tratar_vacina("../data/raw/triplice_viral.csv", "Tríplice viral")
df_pentavalente = tratar_vacina("../data/raw/Pentavalente .csv", "Pentavalente")

df_final = pd.concat([df_bcg, df_poliomielite, df_triplice_viral, df_pentavalente], ignore_index=True)

print(df_final.shape)
df_final.head(10)

(1120, 5)


,codigo_uf,uf,ano,cobertura,vacina
0,11,Rondônia,2013,108.15,BCG
1,12,Acre,2013,106.16,BCG
2,13,Amazonas,2013,116.92,BCG
3,14,Roraima,2013,94.17,BCG
4,15,Pará,2013,117.68,BCG
5,16,Amapá,2013,117.07,BCG
6,17,Tocantins,2013,92.02,BCG
7,21,Maranhão,2013,113.39,BCG
8,22,Piauí,2013,96.50,BCG
9,23,Ceará,2013,108.63,BCG


In [4]:
df_brasil = df_final[df_final["codigo_uf"] == "Total"].copy()
df_brasil["uf"] = "Brasil"
df_brasil = df_brasil.drop(columns=["codigo_uf"])

df_final = df_final[df_final["codigo_uf"] != "Total"].copy()

print("UFs no dataset principal:", df_final["uf"].nunique())
print("Linhas no dataset principal:", df_final.shape)
print("Linhas no total Brasil:", df_brasil.shape)

UFs no dataset principal: 27
Linhas no dataset principal: (1080, 5)
Linhas no total Brasil: (40, 4)


In [5]:
df_final.to_csv("../data/processed/cobertura_vacinal_por_uf.csv", index=False, encoding="utf-8")
df_brasil.to_csv("../data/processed/cobertura_vacinal_brasil.csv", index=False, encoding="utf-8")
print("Arquivos atualizados com sucesso!")


Arquivos atualizados com sucesso!
